# Feature Engineering

Geolocation enrichment + temporal / velocity features for e-commerce fraud data.
Credit-card data is cleaned and saved (PCA features already present).

Outputs → `data/processed/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data_loader import load_creditcard, load_fraud_data, load_ip_country
from src.features import engineer_fraud_features, select_model_features_fraud
from src.preprocessing import clean_creditcard_data, clean_fraud_data

RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

## E-commerce: clean → geo → features

In [ ]:
fraud_raw = load_fraud_data(raw_dir=RAW)
ip_map = load_ip_country(raw_dir=RAW)

fraud = clean_fraud_data(fraud_raw, ip_country_df=ip_map)
fraud = engineer_fraud_features(fraud)

print("Engineered columns:", [
    c for c in fraud.columns
    if c in {
        "hour_of_day", "day_of_week", "time_since_signup",
        "user_tx_count", "device_tx_count", "user_tx_velocity", "country",
    }
])
fraud[[
    "hour_of_day", "day_of_week", "time_since_signup",
    "user_tx_count", "device_tx_count", "user_tx_velocity", "country", "class"
]].describe(include="all")

In [ ]:
out_path = PROC / "fraud_features.csv"
fraud.to_csv(out_path, index=False)
print("Saved", out_path, "shape=", fraud.shape)

X, y = select_model_features_fraud(fraud)
print("Model feature columns:", list(X.columns))
print("Target distribution:\n", y.value_counts(normalize=True))

### Feature documentation

| Feature | Description |
|---------|-------------|
| `hour_of_day` | Hour of `purchase_time` (0–23) |
| `day_of_week` | Weekday of purchase (0=Mon … 6=Sun) |
| `time_since_signup` | Seconds between signup and purchase |
| `user_tx_count` | Lifetime transactions for the user in the dataset |
| `device_tx_count` | Transactions sharing the same `device_id` |
| `user_tx_velocity` | User tx count / activity span (days, min 1) |
| `country` | From IP range lookup |

Scaling / one-hot encoding are applied in the modeling pipeline (`src/pipeline.py`), not here, so we keep raw engineered values reproducible.

## Credit card: clean & save

In [ ]:
cc = clean_creditcard_data(load_creditcard(raw_dir=RAW))
cc_path = PROC / "creditcard_clean.csv"
cc.to_csv(cc_path, index=False)
print("Saved", cc_path, "shape=", cc.shape)